# ConvoKit introduction

Hands-on exploration of [ConvoKit](https://convokit.cornell.edu/index.html) (Conversational Analysis Toolkit).

ConvoKit provides a scikit-learn–style interface for conversational data: **Corpus**, **Conversation**, **Utterance**, and **Speaker** objects, plus transformers for feature extraction and analysis.

Docs: [architecture](https://convokit.cornell.edu/documentation/architecture.html) · [tutorial](https://convokit.cornell.edu/documentation/tutorial.html) · [datasets](https://convokit.cornell.edu/documentation/datasets.html)

## Setup

From the project root, install the notebook dependency group with uv:

```bash
uv sync --group notebooks
# or: mise run notebooks:sync
```

Register the project kernel (included in `notebooks:sync`), then start JupyterLab:

```bash
mise run notebooks:sync
uv run --group notebooks jupyter lab notebooks/
# or: mise run notebooks:lab
```

In Cursor/VS Code, choose the **Kirigami (uv)** kernel (or the `.venv` Python interpreter)—not **Python 3 (ipykernel)**, which may point at an old broken environment.

This notebook loads **`data/topic_102383_posts.json`** — a Discourse export of [topic 102383](https://discuss.python.org/t/102383) (Wheel Variants announcement thread). No network download required.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import convokit
import pandas as pd
from convokit import Corpus

print("ConvoKit imported:", convokit.__file__)

## Load the Discourse topic corpus

Build a ConvoKit `Corpus` from `data/topic_102383_posts.json` (225 posts, one conversation). Posts without `raw` text use `cooked` HTML. Replies are linked to the opening post (flat thread).

In [ ]:
TOPIC_ID = "102383"
DATA_PATH = Path("../data/topic_102383_posts.json")

posts = json.loads(DATA_PATH.read_text())
op_id = str(posts[0]["id"])


def to_timestamp(iso: str) -> float:
    return datetime.fromisoformat(iso.replace("Z", "+00:00")).timestamp()


utterance_rows = []
for post in posts:
    post_id = str(post["id"])
    post_number = post["post_number"]
    utterance_rows.append(
        {
            "id": post_id,
            "speaker": post["username"],
            "conversation_id": TOPIC_ID,
            "reply_to": post_id if post_number == 1 else op_id,
            "timestamp": to_timestamp(post["created_at"]),
            "text": post["raw"] or post["cooked"],
            "meta.post_number": post_number,
            "meta.reply_count": post["reply_count"],
            "meta.quote_count": post["quote_count"],
            "meta.reads": post["reads"],
            "meta.score": post["score"],
            "meta.trust_level": post.get("trust_level"),
            "meta.user_title": post.get("user_title"),
        }
    )

corpus = Corpus.from_pandas(pd.DataFrame(utterance_rows))
corpus.print_summary_stats()
print(f"Loaded {len(posts)} posts from {DATA_PATH.resolve()}")

## Corpus components

Every corpus has **speakers**, **utterances**, and **conversations**. Primary fields (id, text, timestamp, reply_to) live on the object; extra labels live in `.meta`.

In [ ]:
utt = corpus.random_utterance()
print(utt)
print("\n--- primary fields ---")
print("id:", utt.id)
print("conversation_id:", utt.conversation_id)
print("reply_to:", utt.reply_to)
print("timestamp:", utt.timestamp)
print("speaker:", utt.speaker.id)
print("\ntext (first 300 chars):", (utt.text or "")[:300])

In [ ]:
print("utterance metadata keys:", sorted(utt.meta.keys()))
utt.meta

In [ ]:
convo = corpus.random_conversation()
print(convo)
print("\nconversation metadata:", convo.meta)

## Navigate between components

Objects in a corpus link to each other — useful when tracing threads or speaker behavior.

In [ ]:
utt = corpus.random_utterance()
convo = utt.get_conversation()
speaker = utt.speaker

print(f"Utterance {utt.id} in conversation {convo.id}")
print(f"Speaker {speaker.id} has {len(list(speaker.iter_utterances()))} utterances in this corpus")

## Tabular views

Corpus objects can be exported to pandas DataFrames for familiar exploratory analysis.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)
corpus.get_utterances_dataframe().head()

## Inspect speakers in this thread

In [ ]:
speakers_df = corpus.get_speakers_dataframe()
# ConvoKit puts speaker IDs in the index, not an "id" column.
utterance_counts = corpus.get_utterances_dataframe()["speaker"].value_counts()
(
    speakers_df.assign(utterance_count=speakers_df.index.map(utterance_counts))
    .sort_values("utterance_count", ascending=False)
    .head(10)
)

## Next steps

- Re-fetch with `scripts/discourse_api_fetcher.py` to refresh `data/topic_102383_posts.json`
- Try [transformers](https://convokit.cornell.edu/documentation/features.html) (politeness, linguistic coordination) on this corpus
- Parse `reply_to_post_number` from the API when available for finer reply graphs
- Browse [built-in ConvoKit datasets](https://convokit.cornell.edu/documentation/datasets.html) for comparison corpora